# Bloc 05 - Getaround Analysis 

## 02 - ML

In [14]:
import os

import mlflow
import numpy as np
import optuna
import pandas as pd
import torch
from dotenv import load_dotenv
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

load_dotenv()

True

In [15]:
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print('Nombre de GPUs disponibles :', torch.cuda.device_count())
    print('Device courant :', torch.cuda.current_device())

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using {device} device")

NVIDIA GeForce RTX 3070 Ti Laptop GPU
Nombre de GPUs disponibles : 1
Device courant : 0
Using cuda device


In [16]:
os.environ["MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING"] = "false"
MLFLOW_URI = os.environ["MLFLOW_URI"]
EXPERIMENT_NAME = "getaround"

In [17]:
df_pricing = pd.read_csv("../data/get_around_pricing_project.csv")
print(df_pricing.head())

df_pricing.describe()

   Unnamed: 0 model_key  mileage  engine_power    fuel paint_color  \
0           0   Citroën   140411           100  diesel       black   
1           1   Citroën    13929           317  petrol        grey   
2           2   Citroën   183297           120  diesel       white   
3           3   Citroën   128035           135  diesel         red   
4           4   Citroën    97097           160  diesel      silver   

      car_type  private_parking_available  has_gps  has_air_conditioning  \
0  convertible                       True     True                 False   
1  convertible                       True     True                 False   
2  convertible                      False    False                 False   
3  convertible                       True     True                 False   
4  convertible                       True     True                 False   

   automatic_car  has_getaround_connect  has_speed_regulator  winter_tires  \
0          False                   True     

,Unnamed: 0,mileage,engine_power,rental_price_per_day
count,4843.000000,4.843000e+03,4843.00000,4843.000000
mean,2421.000000,1.409628e+05,128.98823,121.214536
std,1398.198007,6.019674e+04,38.99336,33.568268
min,0.000000,-6.400000e+01,0.00000,10.000000
25%,1210.500000,1.029135e+05,100.00000,104.000000
50%,2421.000000,1.410800e+05,120.00000,119.000000
75%,3631.500000,1.751955e+05,135.00000,136.000000
max,4842.000000,1.000376e+06,423.00000,422.000000


In [18]:
df_pricing.drop(columns=["Unnamed: 0"], inplace=True)
df_pricing = df_pricing[(df_pricing["mileage"] > 0) & (df_pricing["engine_power"] > 0)]

In [19]:
df_pricing.describe()

,mileage,engine_power,rental_price_per_day
count,4.841000e+03,4841.000000,4841.000000
mean,1.410042e+05,128.994010,121.185705
std,6.016901e+04,38.930253,33.502751
min,4.760000e+02,25.000000,10.000000
25%,1.030340e+05,100.000000,104.000000
50%,1.410890e+05,120.000000,119.000000
75%,1.752170e+05,135.000000,136.000000
max,1.000376e+06,423.000000,422.000000


In [7]:
# check for NA
df_pricing.isna().sum()

model_key                    0
mileage                      0
engine_power                 0
fuel                         0
paint_color                  0
car_type                     0
private_parking_available    0
has_gps                      0
has_air_conditioning         0
automatic_car                0
has_getaround_connect        0
has_speed_regulator          0
winter_tires                 0
rental_price_per_day         0
dtype: int64

## Fonctions utiles

In [8]:
def build_preprocessor(num_features, cat_features):
    '''
    Création d'un preprocessor
    
    Paramètres
    ----------
    num_features : list
        Liste des features numériques
    cat_features : list
        Liste des features catégorielles
    '''
    
    # Pipeline for numeric features
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler()),
        ]
    )

    # Pipeline for categorical features
    cat_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore"))
        ]
    )

    # Combine numeric and categorical pipelines
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, num_features),
            ("cat", cat_transformer, cat_features),
        ],
        remainder="drop"
    )

    return preprocessor

In [9]:
def split_and_preprocess(X, y, preprocessor, test_size=0.2, random_state=42):
    '''
    Split des features
    
    Paramètres
    ----------
    X : DataFrame
        Liste des features
    y : Series
        Variable target
    preprocessor : ColumnTransformer
        Le ColumnTransformer utilisé
    test_size : float
        Proportion d'échantillon de test
    random_state : int
        variable pour la reproductibilité
    '''
    
    # Dividing into train and test sets...
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)

    print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")
    print(f"y_train shape: {y_train.shape}, y_test shape: {y_test.shape}")
    
    return X_train, X_test, y_train, y_test, preprocessor

In [10]:
def train_and_log_model(model, X_train, y_train, X_test, y_test, preprocessor, experiment_name="experiment"):
    '''
    Entraine un modèle et l'enregistre dans Mlflow
    
    Paramètres
    ----------
    model : Any
        Modèle utilisé
    X_train : DataFrame
        Dataset d'entrainement
    y_train : Series
        Target d'entrainement
    X_test : DataFrame
        Dataset de test
    y_test : Series
        Target de test
    preprocessor : ColumnTransformer
        Le ColumnTransformer utilisé
    experiment_name : str
        Nom de l'experiment dans Mlflow

    X : DataFrame
        Liste des features
    y : Series
        Variable target
    preprocessor : ColumnTransformer
        Le ColumnTransformer utilisé
    test_size : float
        Proportion d'échantillon de test
    random_state : int
        variable pour la reproductibilité
    '''
    
    mlflow.set_experiment(experiment_name)
    num_features = ["mileage", "engine_power"]

    with mlflow.start_run(run_name=type(model).__name__):
        pipeline = Pipeline(steps=[
            ("preprocess", preprocessor),
            ("model", model),
        ])

        for col in num_features:
            X_train[col] = X_train[col].astype("float64")
            X_test[col]  = X_test[col].astype("float64")
    
        # Fit
        pipeline.fit(X_train, y_train)

        # Predictions
        y_train_pred = pipeline.predict(X_train)
        y_test_pred = pipeline.predict(X_test)

        # Metrics
        r2_train = r2_score(y_train, y_train_pred)
        r2_test = r2_score(y_test, y_test_pred)
        rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
        rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
        mae_train = mean_absolute_error(y_train, y_train_pred)
        mae_test = mean_absolute_error(y_test, y_test_pred)

        # Log model params
        mlflow.log_params(model.get_params())

        # Log metrics
        mlflow.log_metric("r2_train", r2_train)
        mlflow.log_metric("r2_test", r2_test)
        mlflow.log_metric("rmse_train", rmse_train)
        mlflow.log_metric("rmse_test", rmse_test)
        mlflow.log_metric("mae_train", mae_train)
        mlflow.log_metric("mae_test", mae_test)

        print("R2 score on training set : ", r2_train)
        print("R2 score on test set : ", r2_test)

        print("RMSE score on training set : ", rmse_train)
        print("RMSE score on test set : ", rmse_test)

        print("MAE score on training set : ", mae_train)
        print("MAE score on test set : ", mae_test)
               
        if isinstance(X_train, pd.DataFrame):
            input_example = X_train.iloc[:5]
        else:
            input_example = pd.DataFrame(X_train).iloc[:5]

        # Log model
        mlflow.sklearn.log_model(pipeline, 
                                 name=f"{type(model).__name__}_model", 
                                 input_example=input_example)
        
        # Log target name
        target_name = getattr(y_train, "name", "target")
        mlflow.log_param("target_name", target_name)

        if hasattr(model, "best_params_") and hasattr(model, "best_score_"):
            print(f"Best parameters: {model.best_params_}\nBest score: {model.best_score_}")
        
    print(f"Model {type(model).__name__} trained and logged in MLflow.")


## Baseline model

In [11]:
target = 'rental_price_per_day'
num_features = ["mileage", "engine_power"]
cat_features = [
    col for col in df_pricing.columns
    if col != target and col not in num_features
]
print(num_features, cat_features)

['mileage', 'engine_power'] ['model_key', 'fuel', 'paint_color', 'car_type', 'private_parking_available', 'has_gps', 'has_air_conditioning', 'automatic_car', 'has_getaround_connect', 'has_speed_regulator', 'winter_tires']


In [12]:
X = df_pricing.drop(columns=[target])
y = df_pricing[target]

print("...Done.")
print()

print('y : ')
print(y.head())
print()
print('X :')
print(X.head())

...Done.

y : 
0    106
1    264
2    101
3    158
4    183
Name: rental_price_per_day, dtype: int64

X :
  model_key  mileage  engine_power    fuel paint_color     car_type  \
0   Citroën   140411           100  diesel       black  convertible   
1   Citroën    13929           317  petrol        grey  convertible   
2   Citroën   183297           120  diesel       white  convertible   
3   Citroën   128035           135  diesel         red  convertible   
4   Citroën    97097           160  diesel      silver  convertible   

   private_parking_available  has_gps  has_air_conditioning  automatic_car  \
0                       True     True                 False          False   
1                       True     True                 False          False   
2                      False    False                 False          False   
3                       True     True                 False          False   
4                       True     True                 False          False   

In [13]:
mlflow.set_tracking_uri(MLFLOW_URI)

In [14]:
preprocessor = build_preprocessor(num_features, cat_features)

In [15]:
X_train, X_test, y_train, y_test, preprocessor_fit = split_and_preprocess(X, y, preprocessor)

X_train shape: (3873, 13), X_test shape: (969, 13)
y_train shape: (3873,), y_test shape: (969,)


In [16]:
# Train model
print("Train model...")
model = LinearRegression()

train_and_log_model(model, X_train, y_train, X_test, y_test, preprocessor_fit, EXPERIMENT_NAME)

Train model...


c:\Apps\anaconda3\envs\cuda\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


R2 score on training set :  0.7177956292836571
R2 score on test set :  0.6458368012695533
RMSE score on training set :  17.77117514878365
RMSE score on test set :  20.2395807544259
MAE score on training set :  11.936734449366192
MAE score on test set :  13.177184980451583


🏃 View run LinearRegression at: https://jiro99-mlflow2.hf.space/#/experiments/6/runs/816030e37b834835bebcc247235f3ea9
🧪 View experiment at: https://jiro99-mlflow2.hf.space/#/experiments/6
Model LinearRegression trained and logged in MLflow.


## Ridge

In [17]:
# Train model
print("Train model...")
model = Ridge(alpha=0.5)

train_and_log_model(model, X_train, y_train, X_test, y_test, preprocessor_fit, EXPERIMENT_NAME)

Train model...


c:\Apps\anaconda3\envs\cuda\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


R2 score on training set :  0.7169481584607391
R2 score on test set :  0.6740365331740613
RMSE score on training set :  17.79783891461625
RMSE score on test set :  19.41709522925772
MAE score on training set :  11.960951028405878
MAE score on test set :  13.038912481102495


🏃 View run Ridge at: https://jiro99-mlflow2.hf.space/#/experiments/6/runs/139f7467115c4e12b46347638f619385
🧪 View experiment at: https://jiro99-mlflow2.hf.space/#/experiments/6
Model Ridge trained and logged in MLflow.


In [18]:
# Train model
print("Train model...")
model = Ridge()
params = {'alpha': [0.01, 0.05, 0.1, 0.5, 1, 5, 10, 50]}

grid_search = GridSearchCV(model, params, cv=3)
train_and_log_model(grid_search, X_train, y_train, X_test, y_test, preprocessor_fit, EXPERIMENT_NAME)

Train model...


c:\Apps\anaconda3\envs\cuda\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


R2 score on training set :  0.7176060010315034
R2 score on test set :  0.6597381009004044
RMSE score on training set :  17.77714484896366
RMSE score on test set :  19.838391532864584
MAE score on training set :  11.948441363625687
MAE score on test set :  13.119204982510004


Best parameters: {'alpha': 0.1}
Best score: 0.707087697548968
🏃 View run GridSearchCV at: https://jiro99-mlflow2.hf.space/#/experiments/6/runs/2e880371c32f4a91bc50d6df257331de
🧪 View experiment at: https://jiro99-mlflow2.hf.space/#/experiments/6
Model GridSearchCV trained and logged in MLflow.


In [19]:
grid_search.best_params_

# Train model
print("Train model...")
model = Ridge(**grid_search.best_params_)

train_and_log_model(model, X_train, y_train, X_test, y_test, preprocessor_fit, EXPERIMENT_NAME)

Train model...


c:\Apps\anaconda3\envs\cuda\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


R2 score on training set :  0.7176060010315034
R2 score on test set :  0.6597381009004044
RMSE score on training set :  17.77714484896366
RMSE score on test set :  19.838391532864584
MAE score on training set :  11.948441363625687
MAE score on test set :  13.119204982510004


🏃 View run Ridge at: https://jiro99-mlflow2.hf.space/#/experiments/6/runs/7936fc0479664e94a359b962ebc3c59b
🧪 View experiment at: https://jiro99-mlflow2.hf.space/#/experiments/6
Model Ridge trained and logged in MLflow.


## Lasso

In [20]:
# Train model
print("Train model...")
model = Lasso(alpha=0.5)

train_and_log_model(model, X_train, y_train, X_test, y_test, preprocessor_fit, EXPERIMENT_NAME)

Train model...


c:\Apps\anaconda3\envs\cuda\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


R2 score on training set :  0.6656817606970497
R2 score on test set :  0.6395797033560208
RMSE score on training set :  19.342576127344643
RMSE score on test set :  20.41758707427543
MAE score on training set :  13.065819348129862
MAE score on test set :  13.954438366426393


🏃 View run Lasso at: https://jiro99-mlflow2.hf.space/#/experiments/6/runs/470113a1c744434d9f311be15405c2f6
🧪 View experiment at: https://jiro99-mlflow2.hf.space/#/experiments/6
Model Lasso trained and logged in MLflow.


In [21]:
# Train model
print("Train model...")
model = Lasso()
params = {'alpha': [0.005, 0.008, 0.01, 0.05, 0.1, 0.5, 1, 5, 10, 50]}

grid_search = GridSearchCV(model, params, cv=3)
# grid_search.fit(X_train, y_train)

train_and_log_model(grid_search, X_train, y_train, X_test, y_test, preprocessor_fit, EXPERIMENT_NAME)

print(grid_search.best_estimator_)

Train model...


c:\Apps\anaconda3\envs\cuda\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


R2 score on training set :  0.7166589965317718
R2 score on test set :  0.6723127034285887
RMSE score on training set :  17.80692760961849
RMSE score on test set :  19.46837034628615
MAE score on training set :  11.971549281020435
MAE score on test set :  13.049360433320226


Best parameters: {'alpha': 0.005}
Best score: 0.7067165750574391
🏃 View run GridSearchCV at: https://jiro99-mlflow2.hf.space/#/experiments/6/runs/3fba745e2be44ae0bcb408f91cbc186c
🧪 View experiment at: https://jiro99-mlflow2.hf.space/#/experiments/6
Model GridSearchCV trained and logged in MLflow.
Lasso(alpha=0.005)


In [22]:
grid_search.best_params_

# Train model
print("Train model...")
model = Lasso(**grid_search.best_params_)

train_and_log_model(model, X_train, y_train, X_test, y_test, preprocessor_fit, EXPERIMENT_NAME)

Train model...


c:\Apps\anaconda3\envs\cuda\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


R2 score on training set :  0.7166589965317718
R2 score on test set :  0.6723127034285887
RMSE score on training set :  17.80692760961849
RMSE score on test set :  19.46837034628615
MAE score on training set :  11.971549281020435
MAE score on test set :  13.049360433320226


🏃 View run Lasso at: https://jiro99-mlflow2.hf.space/#/experiments/6/runs/581e855eed1d48abae3fe7c6700b989a
🧪 View experiment at: https://jiro99-mlflow2.hf.space/#/experiments/6
Model Lasso trained and logged in MLflow.


## Optuna

In [23]:
def objective(trial):
    optuna_param_grid = {
        'n_estimators': trial.suggest_int('n_estimators', 10, 200),
        'max_depth': trial.suggest_categorical('max_depth', [5, 8, 10, 15]),
        'min_samples_split': trial.suggest_int('min_samples_split', 3, 10),
        'min_samples_leaf': trial.suggest_categorical('min_samples_leaf', [2, 5, 10]),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'criterion': trial.suggest_categorical('criterion', ['squared_error', 'poisson', 'friedman_mse', 'absolute_error']),
        #'class_weight': trial.suggest_categorical('class_weight', [None, 'balanced', 'balanced_subsample']),
        'random_state': 42
        }
    
    classifier = RandomForestRegressor(**optuna_param_grid)

    pipeline = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", classifier),
    ])

    pipeline.fit(X_train, y_train)

    return pipeline.score(X_test, y_test)

param_grid = {

}

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2026-03-10 00:14:34,840] A new study created in memory with name: no-name-947cb131-2737-4b03-b6d0-78c82908a10b
c:\Apps\anaconda3\envs\cuda\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
[I 2026-03-10 00:14:35,987] Trial 0 finished with value: 0.6690768574225809 and parameters: {'n_estimators': 112, 'max_depth': 15, 'min_samples_split': 3, 'min_samples_leaf': 2, 'max_features': 'log2', 'criterion': 'poisson'}. Best is trial 0 with value: 0.6690768574225809.
c:\Apps\anaconda3\envs\cuda\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
[I 2026-03-10 00:14:36,607] Trial 1 finished with value: 0.555067615876635 and parameters: {'n_estimators': 145, 'max_depth': 8, 'min_samples_split': 4, 'm

In [24]:
print(f"Best parameters: {study.best_params}\nBest score: {study.best_value}")

Best parameters: {'n_estimators': 71, 'max_depth': 15, 'min_samples_split': 4, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'criterion': 'squared_error'}
Best score: 0.7126739959956576


In [25]:
print("Train model...")
best_model = RandomForestRegressor(**study.best_params)#.fit(X_train, y_train)

train_and_log_model(best_model, X_train, y_train, X_test, y_test, preprocessor_fit, EXPERIMENT_NAME)

Train model...


c:\Apps\anaconda3\envs\cuda\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


R2 score on training set :  0.8160850290379795
R2 score on test set :  0.7014512565043795
RMSE score on training set :  14.34639621193818
RMSE score on test set :  18.5826402963436
MAE score on training set :  9.370995956080097
MAE score on test set :  12.176908298864227


🏃 View run RandomForestRegressor at: https://jiro99-mlflow2.hf.space/#/experiments/6/runs/13153f6fd9914009964a6832243d4811
🧪 View experiment at: https://jiro99-mlflow2.hf.space/#/experiments/6
Model RandomForestRegressor trained and logged in MLflow.


## XGBoost

In [27]:
from sklearn.ensemble import GradientBoostingRegressor

model = GradientBoostingRegressor(
)

params = {
    "learning_rate": [0.01, 0.05, 0.08, 0.1, 0.3, 0.5, 0.8],
    "n_estimators": [2, 5, 10, 20, 30, 50, 100, 200, 300, 400, 500],
    "min_samples_leaf": [1, 3, 5, 8, 10],
    "max_depth": [1, 3, 5, 10],
    "random_state": [42]

}

gridsearch = GridSearchCV(
    model, param_grid=params, cv=3
)

train_and_log_model(gridsearch, X_train, y_train, X_test, y_test, preprocessor_fit, EXPERIMENT_NAME)

print("...Done.")
print("Best hyperparameters : ", gridsearch.best_params_)
print("Best validation accuracy : ", gridsearch.best_score_)

c:\Apps\anaconda3\envs\cuda\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


R2 score on training set :  0.8994867493347365
R2 score on test set :  0.7720155451377271
RMSE score on training set :  10.605861187183649
RMSE score on test set :  16.238743214350677
MAE score on training set :  7.232505390678555
MAE score on test set :  10.43995716147173


Best parameters: {'learning_rate': 0.05, 'max_depth': 5, 'min_samples_leaf': 1, 'n_estimators': 400, 'random_state': 42}
Best score: 0.7581266867503786
🏃 View run GridSearchCV at: https://jiro99-mlflow2.hf.space/#/experiments/6/runs/a969f0d1f75a4f65b2b5450ba20ef26d
🧪 View experiment at: https://jiro99-mlflow2.hf.space/#/experiments/6
Model GridSearchCV trained and logged in MLflow.
...Done.
Best hyperparameters :  {'learning_rate': 0.05, 'max_depth': 5, 'min_samples_leaf': 1, 'n_estimators': 400, 'random_state': 42}
Best validation accuracy :  0.7581266867503786


In [64]:
print("Train model...")
best_model = GradientBoostingRegressor(**gridsearch.best_params_)

train_and_log_model(best_model, X_train, y_train, X_test, y_test, preprocessor_fit, EXPERIMENT_NAME)

Train model...


c:\Apps\anaconda3\envs\cuda\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


R2 score on training set :  0.8994867493347365
R2 score on test set :  0.7720155451377271
RMSE score on training set :  10.605861187183649
RMSE score on test set :  16.238743214350677
MAE score on training set :  7.232505390678555
MAE score on test set :  10.43995716147173


🏃 View run GradientBoostingRegressor at: https://jiro99-mlflow2.hf.space/#/experiments/6/runs/8b0f709c76ed4f2aaf5a7be268fa3f2d
🧪 View experiment at: https://jiro99-mlflow2.hf.space/#/experiments/6
Model GradientBoostingRegressor trained and logged in MLflow.


Le meilleur modèle est le GradientBoostingRegressor avec les paramètres {'learning_rate': 0.05, 'max_depth': 5, 'min_samples_leaf': 1, 'n_estimators': 400, 'random_state': 42}  

Il obtient le meilleur R2 (0.772) et les plus faibles erreurs de prédiction RMSE (16.23) et MAE (10.43) sur le test set